#  NLP Preprocessing Engine

This notebook contains my complete solution for the NLP Preprocessing Engine assignment.  
I have implemented a modular, production‑style text preprocessing pipeline and validated it using diverse, noisy real‑world‑like examples.

---

## Task 1 – Conceptual Understanding

### 1. Difference between "Love" and "love" in NLP

In normal reading, "Love" and "love" look almost the same, but in NLP, the difference in case can carry useful information.  
"Love" (capital L) might appear at the start of a sentence, in a title, or as part of a proper noun (for example, a song or movie name), while "love" (lowercase) usually represents the generic verb or noun.  
Many preprocessing pipelines convert everything to lowercase so that "Love", "LOVE", and "love" all map to one normalized token, which simplifies the vocabulary and lets the model focus on meaning rather than surface case differences.  
However, before lowercasing, the original case can encode emphasis or named entities, so there is a trade‑off between simplicity and preserving that extra signal.

---

### 2. What happens if stopwords are not removed?

If stopwords like "the", "is", "at", "on", "and", "of" are not removed, they tend to dominate the corpus because they occur very frequently.  
This increases the size of the vocabulary and can introduce noise when building bag‑of‑words or TF‑IDF representations, since a large part of the feature space is occupied by low‑information words.  
Models may become slower to train and may pay less attention to informative content words because high‑frequency stopwords overshadow them in counts.  
In some tasks (like simple document classification), removing stopwords improves efficiency and can slightly improve performance, but for other tasks it may be better to keep them.

---

### 3. Two real‑world scenarios where removing stopwords can be harmful

1. **Sentiment and opinion analysis**  
   Words like "no", "not", "never", "without" can completely flip the sentiment or meaning of a sentence.  
   If we remove them as generic stopwords, "I am not happy" becomes close to "I am happy", and "This is no good" looks similar to "This is good", which is dangerous for sentiment models.

2. **Question answering, search queries, and logical constraints**  
   Interrogative words and function words such as "who", "when", "where", "how", "if", "until", "with", "without" often appear in stopword lists but are semantically important.  
   For example, "students with scholarship" vs "students without scholarship" or "who discovered X" vs "when was X discovered" — dropping these words erases the real intent and conditions of the query.

---

### 4. Difference between stemming and lemmatization

**Stemming** is a rule‑based process that chops off prefixes and suffixes to get a stem, without caring whether the result is a valid word.  
For example, "studies", "studying", and "studied" might all be reduced to "studi", and "better" might become "bett".  
It is fast and simple, but the output can be unnatural and sometimes merges words that should stay distinct.

**Lemmatization** is a more linguistically informed process that uses a vocabulary and, often, part‑of‑speech information to map words to their correct base form (lemma).  
For example, "studies", "studying", and "studied" all become "study", "went" becomes "go", and "better" as an adjective becomes "good".  
Lemmatization is slower and more resource‑heavy than stemming, but the resulting tokens are real dictionary words and are usually better for interpretability and many downstream tasks.

## Task 2 – Preprocessing Function

In [1]:
import re
from statistics import mean

# --- Precompiled regular expressions for efficiency and clarity ---

URL_PATTERN = re.compile(
    r"(?i)\b((?:https?:\/\/|www\.)[^\s]+)"
)
EMAIL_PATTERN = re.compile(
    r"(?i)\b[\w.+-]+@[\w-]+\.[\w.-]+\b"
)
NUMBER_PATTERN = re.compile(
    r"(?<!\w)\d+[.,:/\-]?\d*(?!\w)"
)
REPEATED_CHARS_PATTERN = re.compile(r"(.)\1{2,}")  # any char repeated 3+ times

# Tokens with length <= 2 that we still want to keep
SHORT_TOKEN_WHITELIST = {"no", "not", "ok", "omg"}


def clean_whitespace(text: str) -> str:
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def normalize_repeated_chars(token: str) -> str:
    # Step 1: collapse 3+ to 2
    token = REPEATED_CHARS_PATTERN.sub(r"\1\1", token)

    # Step 2: shrink double consonants
    result_chars = []
    for i, ch in enumerate(token):
        if i > 0 and ch == token[i - 1]:
            # if this is a double consonant, keep only one
            if ch.lower() not in "aeiou":
                continue
        result_chars.append(ch)

    return "".join(result_chars)


def preprocess_text(text):
    # 1. Robust input handling
    if text is None:
        text = ""
    if not isinstance(text, str):
        text = str(text)

    # 2. Normalize whitespace early
    text = clean_whitespace(text)

    # 3. Remove URLs and emails
    text = URL_PATTERN.sub(" ", text)
    text = EMAIL_PATTERN.sub(" ", text)

    # 4. Remove standalone numbers (scores, phone numbers, etc.)
    text = NUMBER_PATTERN.sub(" ", text)

    # 5. Put space around punctuation so it does not stick to words
    text = re.sub(r"([!?.,;:/()\"'])", r" \1 ", text)
    text = clean_whitespace(text)

    # 6. Lowercase
    text = text.lower()

    # 7. Basic tokenization on whitespace
    raw_tokens = text.split()

    processed_tokens = []
    for tok in raw_tokens:
        # Strip leading/trailing punctuation like !!!, ???, etc.
        tok = tok.strip(".,!?;:'\"()[]{}<>-_/\\|")

        if not tok:
            continue

        # Normalize repeated characters inside the token
        tok = normalize_repeated_chars(tok)

        if not tok:
            continue

        # Remove short tokens except whitelisted ones
        if len(tok) <= 2 and tok not in SHORT_TOKEN_WHITELIST:
            continue

        processed_tokens.append(tok)

    # 8. Build clean sentence and normalize whitespace again
    clean_sentence = clean_whitespace(" ".join(processed_tokens))

    return processed_tokens, clean_sentence

## Task 3 – Stress Testing the Preprocessing Function

Below, I test the `preprocess_text` function on at least ten diverse sentences that include:
- emojis
- numbers
- slang and informal writing
- repeated characters
- URLs
- mixed casing

For each sentence, I display:
- Original text  
- Cleaned tokens  
- Cleaned sentence

In [2]:
sample_sentences = [
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product 😍😍",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Visit https://openai.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this"
]

analysis_results = []

for idx, sent in enumerate(sample_sentences, start=1):
    tokens, clean_sent = preprocess_text(sent)
    result = {
        "id": idx,
        "original": sent,
        "tokens": tokens,
        "clean_sentence": clean_sent
    }
    analysis_results.append(result)

# Pretty print each result
for row in analysis_results:
    print(f"Sentence {row['id']}")
    print("Original:        ", row["original"])
    print("Cleaned tokens:  ", row["tokens"])
    print("Cleaned sentence:", row["clean_sentence"])
    print("-" * 80)

Sentence 1
Original:         Get 100% FREE access now!!!
Cleaned tokens:   ['get', 'free', 'aces', 'now']
Cleaned sentence: get free aces now
--------------------------------------------------------------------------------
Sentence 2
Original:         I absolutely looooved this product 😍😍
Cleaned tokens:   ['absolutely', 'looved', 'this', 'product']
Cleaned sentence: absolutely looved this product
--------------------------------------------------------------------------------
Sentence 3
Original:         Worst service ever... 0/10
Cleaned tokens:   ['worst', 'service', 'ever']
Cleaned sentence: worst service ever
--------------------------------------------------------------------------------
Sentence 4
Original:         Call me at 9876543210
Cleaned tokens:   ['cal']
Cleaned sentence: cal
--------------------------------------------------------------------------------
Sentence 5
Original:         This is THE best course!!!
Cleaned tokens:   ['this', 'the', 'best', 'course']
Cleaned s

## Task 4 - Token Analytics

In [3]:
def sentence_token_stats(tokens):
    if not tokens:
        return {
            "total_tokens": 0,
            "unique_tokens": 0,
            "avg_token_length": 0.0
        }

    lengths = [len(t) for t in tokens]
    return {
        "total_tokens": len(tokens),
        "unique_tokens": len(set(tokens)),
        "avg_token_length": round(mean(lengths), 2)
    }


# Attach stats to each sentence
for row in analysis_results:
    stats = sentence_token_stats(row["tokens"])
    row.update(stats)

import pandas as pd

df_results = pd.DataFrame(analysis_results)
df_results

,id,original,tokens,clean_sentence,total_tokens,unique_tokens,avg_token_length
0,1,Get 100% FREE access now!!!,"[get, free, aces, now]",get free aces now,4,4,3.50
1,2,I absolutely looooved this product 😍😍,"[absolutely, looved, this, product]",absolutely looved this product,4,4,6.75
2,3,Worst service ever... 0/10,"[worst, service, ever]",worst service ever,3,3,5.33
3,4,Call me at 9876543210,[cal],cal,1,1,3.00
4,5,This is THE best course!!!,"[this, the, best, course]",this the best course,4,4,4.25
5,6,Visit https://openai.com now!,"[visit, now]",visit now,2,2,4.00
6,7,Nooooo this is baaad!!!,"[noo, this, baad]",noo this baad,3,3,3.67
7,8,OK OK OK I got it,"[ok, ok, ok, got]",ok ok ok got,4,2,2.25
8,9,Win $$$ now!!! Limited offer!!!,"[win, now, limited, ofer]",win now limited ofer,4,4,4.25
9,10,I am not happy with this,"[not, hapy, with, this]",not hapy with this,4,4,3.75


## Task 4 – Token Analytics and Interpretation

Using the statistics above, I compute for each sentence:
- total number of tokens
- number of unique tokens
- average token length

### 1. Which sentence had the most noise?

I consider a sentence to be "noisy" if a large portion of its original content is removed or heavily normalized (for example, many digits, URLs, random characters, or long repetitions).  
From the token statistics and the before/after inspection, the sentence:

> "Call me at 9876543210"

has very little meaningful linguistic content apart from filler words and a phone number.  
The phone number is completely removed as non‑linguistic numeric data, leaving almost no substantive tokens, so this sentence has the highest proportion of noise compared to useful text in this dataset.


### 2. Which sentence retained the most meaningful tokens after cleaning?

A sentence retains more meaningful tokens if, after cleaning, most words still carry semantic information that would be useful to a model (especially for sentiment or intent).  
In this set, a good example is:

> "I am not happy with this"

After preprocessing, the key tokens like "not" and "happy" are preserved, and the sentence structure remains understandable.  
This shows that the pipeline successfully removes noise (if any) while keeping short but crucial tokens (like "not") thanks to the whitelist, which is important for correct sentiment interpretation.

## Task 5 – Frequency Analysis

In [4]:
from collections import Counter

# Combine all tokens from all processed sentences
all_tokens = []
for row in analysis_results:
    all_tokens.extend(row["tokens"])

token_freq = Counter(all_tokens)

top_10_most = token_freq.most_common(10)

# For least frequent, sort ascending by count then by token
least_5 = sorted(token_freq.items(), key=lambda x: (x[1], x[0]))[:5]

print("Top 10 most frequent words:")
for word, count in top_10_most:
    print(f"{word}: {count}")

print("\nTop 5 least frequent words:")
for word, count in least_5:
    print(f"{word}: {count}")

Top 10 most frequent words:
this: 4
now: 3
ok: 3
get: 1
free: 1
aces: 1
absolutely: 1
looved: 1
product: 1
worst: 1

Top 5 least frequent words:
absolutely: 1
aces: 1
baad: 1
best: 1
cal: 1


## Task 5 – Frequency Analysis Discussion

After combining tokens from all sentences and computing word frequencies:

- The **top 10 most frequent words** highlight tokens that appear repeatedly across different noisy inputs, such as common sentiment words or action words
  These words tend to dominate the small corpus and would be important features for downstream models.

- The **top 5 least frequent words** are tokens that occur rarely (often only once).  
  They may correspond to more specific or context‑dependent terms and can be candidates for special handling

## Task 6 – Full Pipeline

In [5]:
def full_pipeline(text_list):
    if text_list is None:
        text_list = []

    if not isinstance(text_list, (list, tuple)):
        raise TypeError("text_list must be a list or tuple of strings")

    all_tokens = []
    all_clean_sentences = []

    for text in text_list:
        tokens, clean_sentence = preprocess_text(text)
        all_tokens.append(tokens)
        all_clean_sentences.append(clean_sentence)

    return {
        "tokens": all_tokens,
        "clean_sentences": all_clean_sentences
    }


# Demonstration on the sample sentences
pipeline_output = full_pipeline(sample_sentences)
pipeline_output

{'tokens': [['get', 'free', 'aces', 'now'],
  ['absolutely', 'looved', 'this', 'product'],
  ['worst', 'service', 'ever'],
  ['cal'],
  ['this', 'the', 'best', 'course'],
  ['visit', 'now'],
  ['noo', 'this', 'baad'],
  ['ok', 'ok', 'ok', 'got'],
  ['win', 'now', 'limited', 'ofer'],
  ['not', 'hapy', 'with', 'this']],
 'clean_sentences': ['get free aces now',
  'absolutely looved this product',
  'worst service ever',
  'cal',
  'this the best course',
  'visit now',
  'noo this baad',
  'ok ok ok got',
  'win now limited ofer',
  'not hapy with this']}

## Task 6 – Full Pipeline Description

The `full_pipeline` function wraps the `preprocess_text` function to process an entire list of sentences in one call.  
It returns:
- `tokens`: a list where each element is the list of cleaned tokens for one input sentence  
- `clean_sentences`: a list of cleaned sentence strings aligned with the original order

This structure makes it easy to integrate the preprocessing engine into a downstream machine learning workflow, since models can consume either token lists or fully cleaned sentences.

## Task 7 – Error Handling and Edge Cases

In [9]:
edge_cases = [
    "",              # empty string
    "😂😂😂🔥🔥",     # only emojis
    "1234567890",    # only numbers
    None             # None instead of string
]

for idx, ec in enumerate(edge_cases, start=1):
    tokens, clean_sent = preprocess_text(ec)
    print(f"Edge case {idx}: input = {repr(ec)}")
    print("  Tokens:        ", tokens)
    print("  Clean sentence:", repr(clean_sent))
    print("-" * 60)

Edge case 1: input = ''
  Tokens:         []
  Clean sentence: ''
------------------------------------------------------------
Edge case 2: input = '😂😂😂🔥🔥'
  Tokens:         []
  Clean sentence: ''
------------------------------------------------------------
Edge case 3: input = '1234567890'
  Tokens:         []
  Clean sentence: ''
------------------------------------------------------------
Edge case 4: input = None
  Tokens:         []
  Clean sentence: ''
------------------------------------------------------------


## Task 7 – Error Handling

I explicitly tested the preprocessing function on several difficult edge cases:

1. **Empty string (`""`)**  
   The function returns an empty token list and an empty cleaned sentence string.  
   This shows that the pipeline does not crash or behave unexpectedly when there is no text.

2. **Only emojis (`"😂😂😂🔥🔥"`)**  
   Emojis are effectively removed during the normalization and filtering steps, producing an empty token list.  
   This prevents non‑alphanumeric symbols from polluting the vocabulary in traditional word‑based models.

3. **Only numbers (`"1234567890"`)**  
   The numeric pattern removes the digits, resulting in no tokens and an empty cleaned sentence.  
   This behavior is desirable when numbers like phone numbers or scores are treated as noise for text‑only models.

4. **`None` input**  
   Passing `None` does not raise an error; the function safely converts it to an empty string and processes it.  
   This makes the preprocessing engine more robust to missing values in real datasets, where some text fields may be null.

Overall, these tests demonstrate that the preprocessing engine can handle noisy, incomplete, or unusual inputs gracefully without throwing exceptions.